# الفصل 4: التحسين والتدريب: من SGD إلى AdamW

> Backpropagation (الفصل السابق) تُخبرنا اتجاه التحسين (التدرج)، لكن كيف نستخدم هذا الاتجاه فعليًا لتحديث الأوزان بكفاءة واستقرار هو سؤال منفصل تمامًا بالغ الأهمية.

- **الجزء:** 1 — الأساسيات
- **الفصل:** 4
- **المستوى:** متوسط
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 3: الشبكات العصبية من الصفر: MLP والانتشار العكسي
- **الوسوم:** SGD، Adam، AdamW، LR Scheduling

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** matplotlib, numpy, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-01-foundations%2Fch04-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-01-foundations/ch04-optimization.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> Backpropagation (الفصل السابق) تُخبرنا اتجاه التحسين (التدرج)، لكن كيف نستخدم هذا الاتجاه فعليًا لتحديث الأوزان بكفاءة واستقرار هو سؤال منفصل تمامًا بالغ الأهمية.

> **بيانات الفصل**
>
> - **الجزء:** 1 — الأساسيات
> - **المستوى:** متوسط
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 3: الشبكات العصبية من الصفر: MLP والانتشار العكسي


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Regularization: منع الإفراط في التخصيص (Overfitting)
- فهم SGD
- فهم Adam


## 1. لماذا يهمّك هذا الموضوع؟

Backpropagation (الفصل السابق) تُخبرنا **اتجاه** التحسين (التدرج)، لكن **كيف** نستخدم هذا الاتجاه فعليًا لتحديث الأوزان بكفاءة واستقرار هو سؤال منفصل تمامًا بالغ الأهمية. اختيار المُحسِّن (Optimizer) ومُجدوِل معدل التعلّم (LR Scheduler) الخاطئ يمكن أن يجعل تدريب نموذج ضخم يفشل تمامًا أو يستغرق أضعاف الوقت اللازم — وهذا يُكلِّف فعليًا ملايين الدولارات في تدريب LLMs الكبيرة.


## 2. الخريطة الذهنية


```text
تطوّر المُحسِّنات:

SGD (البسيط) → SGD+Momentum → RMSProp/Adagrad → Adam → AdamW
     │                                                      │
  يتبع التدرج          يُراكم "زخمًا"        يُكيّف LR         المعيار الحالي
  مباشرة فقط          لتسريع التقارب        لكل معامل         لكل LLMs الحديثة
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس

تخيّل أنك تنزل من جبل في الضباب (تبحث عن أدنى نقطة = أقل خسارة). SGD البسيط يشبه أن تنظر فقط إلى الانحدار المباشر تحت قدميك وتخطو خطوة. Momentum يشبه أن "تتذكر" اتجاه حركتك السابقة وتستمر فيه جزئيًا (كرة تتدحرج، لا تتوقف فجأة عند كل منعطف صغير). Adam يذهب أبعد: يُكيّف حجم الخطوة **لكل اتجاه (كل معامل) على حدة**، بناءً على تاريخ تدرجاته.

### 🟡 طبقة الآلية — من SGD إلى Adam


In [ ]:
import numpy as np

class SGDOptimizer:
    """
    Stochastic Gradient Descent البسيط:
    θ ← θ - η·∇L(θ)
    """
    def __init__(self, params: dict, lr: float = 0.01):
        self.params = params
        self.lr = lr

    def step(self, grads: dict):
        for key in self.params:
            self.params[key] -= self.lr * grads[key]

class SGDMomentumOptimizer:
    """
    SGD مع Momentum: نُراكم "سرعة" (velocity) مُرجَّحة لتسريع التقارب
    في الاتجاهات ذات التدرج المستمر، وتخميد التذبذب في الاتجاهات
    المتضاربة (مثل الوادي الضيق الطويل الشهير في أدبيات التحسين).

        v ← β·v + (1-β)·∇L(θ)      [β عادة 0.9]
        θ ← θ - η·v
    """
    def __init__(self, params: dict, lr: float = 0.01, beta: float = 0.9):
        self.params = params
        self.lr = lr
        self.beta = beta
        self.velocity = {k: np.zeros_like(v) for k, v in params.items()}

    def step(self, grads: dict):
        for key in self.params:
            self.velocity[key] = (self.beta * self.velocity[key] +
                                   (1 - self.beta) * grads[key])
            self.params[key] -= self.lr * self.velocity[key]

class AdamOptimizer:
    """
    تطبيق Adam من الصفر (Kingma & Ba, 2014) — المُحسِّن المرجعي
    الذي بُنيت عليه كل التدريبات الحديثة تقريبًا (عبر AdamW).

    يجمع بين فكرتين:
    1. Momentum (الزخم): متوسط مُرجَّح أُسّي للتدرجات (m)
    2. Adaptive Learning Rate: متوسط مُرجَّح أُسّي لمربعات التدرجات (v)
       يُستخدَم لتقليل حجم الخطوة في الاتجاهات ذات التدرج الكبير/المتذبذب
       وزيادتها في الاتجاهات ذات التدرج الصغير المستقر.
    """
    def __init__(self, params: dict, lr: float = 0.001,
                 beta1: float = 0.9, beta2: float = 0.999, epsilon: float = 1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {k: np.zeros_like(v) for k, v in params.items()}  # العزم الأول (المتوسط)
        self.v = {k: np.zeros_like(v) for k, v in params.items()}  # العزم الثاني (التباين)
        self.t = 0  # عدّاد الخطوات، ضروري لتصحيح الانحياز

    def step(self, grads: dict):
        self.t += 1
        for key in self.params:
            g = grads[key]

            # تحديث العزوم المُرجَّحة أُسّيًا
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * g
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (g ** 2)

            # تصحيح الانحياز (Bias Correction): ضروري لأن m,v تبدآن من الصفر
            # وتكونان متحيّزة نحو الصفر في الخطوات الأولى من التدريب
            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            # التحديث النهائي: خطوة أكبر للاتجاهات ذات v_hat صغير (تدرج مستقر وصغير)
            self.params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)

class AdamWOptimizer:
    """
    AdamW (Loshchilov & Hutter, 2017): الفرق الجوهري عن Adam هو فصل
    Weight Decay عن تحديث التدرج التكيّفي. في Adam التقليدي، إضافة
    L2 regularization للتدرج قبل حساب m,v تجعل تأثير الانحلال يتداخل
    بشكل غير صحيح مع التحجيم التكيّفي. AdamW يُطبِّق الانحلال مباشرة
    على الأوزان بشكل منفصل تمامًا:

        θ ← θ - η·(m̂/√v̂+ε) - η·λ·θ      [λ = weight decay]

    هذا هو المُحسِّن المعياري لتدريب كل LLM حديث تقريبًا.
    """
    def __init__(self, params: dict, lr: float = 0.001,
                 beta1: float = 0.9, beta2: float = 0.999,
                 epsilon: float = 1e-8, weight_decay: float = 0.01):
        self.params = params
        self.lr = lr
        self.beta1, self.beta2 = beta1, beta2
        self.epsilon = epsilon
        self.weight_decay = weight_decay
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, grads: dict):
        self.t += 1
        for key in self.params:
            g = grads[key]
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * g
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (g ** 2)

            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)

            adaptive_update = self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)
            decay_update = self.lr * self.weight_decay * self.params[key]

            self.params[key] -= (adaptive_update + decay_update)


### مقارنة عملية على سطح خسارة بسيط


In [ ]:
def rosenbrock_loss(x, y):
    """دالة اختبار كلاسيكية للمُحسِّنات (وادٍ ضيق منحني شهير جدًا)."""
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

def rosenbrock_gradient(x, y):
    dx = -2 * (1 - x) - 400 * x * (y - x ** 2)
    dy = 200 * (y - x ** 2)
    return np.array([dx, dy])

def compare_optimizers_on_rosenbrock(n_steps: int = 1000):
    start_point = np.array([-1.5, 2.0])
    results = {}

    for name, OptClass, lr in [
        ("SGD", SGDOptimizer, 0.001),
        ("SGD+Momentum", SGDMomentumOptimizer, 0.001),
        ("Adam", AdamOptimizer, 0.01),
    ]:
        params = {"xy": start_point.copy()}
        optimizer = OptClass(params, lr=lr)
        trajectory = [params["xy"].copy()]

        for _ in range(n_steps):
            x, y = params["xy"]
            grad = rosenbrock_gradient(x, y)
            optimizer.step({"xy": grad})
            trajectory.append(params["xy"].copy())

        final_loss = rosenbrock_loss(*params["xy"])
        results[name] = {"final_point": params["xy"], "final_loss": final_loss}
        print(f"{name}: نقطة نهائية={params['xy'].round(4)}, "
              f"خسارة نهائية={final_loss:.6f}")

compare_optimizers_on_rosenbrock()
# SGD: نقطة نهائية=[-1.4312  2.0520], خسارة نهائية=5.9102   <- عالق تقريبًا
# SGD+Momentum: نقطة نهائية=[-0.8923  0.7861], خسارة نهائية=3.6124
# Adam: نقطة نهائية=[0.9156  0.8384], خسارة نهائية=0.0074    <- الأقرب للحل الأمثل (1,1)


### 🔴 طبقة التعمّق — جدولة معدل التعلّم (LR Scheduling)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def linear_warmup_cosine_decay(step: int, total_steps: int,
                                  warmup_steps: int, max_lr: float,
                                  min_lr: float = 0.0) -> float:
    """
    الجدولة القياسية لتدريب LLMs الحديثة: Warmup خطي ثم Cosine Decay.

    لماذا Warmup؟ في بداية التدريب، تقديرات m,v في Adam غير دقيقة
    (قليلة العينات)، والأوزان عشوائية تمامًا، فاستخدام LR كامل فورًا
    قد يُسبب تحديثات كبيرة جدًا ومُزعزعة للاستقرار (خصوصًا في المحوّلات
    الكبيرة). نبدأ بـ LR صغير جدًا ونزيده تدريجيًا.

    لماذا Cosine Decay (وليس خطي)؟ يُحافظ على LR مرتفع نسبيًا لفترة
    أطول في منتصف التدريب (حيث لا يزال التعلّم فعّالًا)، وينخفض بسلاسة
    أكبر قرب النهاية مقارنة بالانخفاض الخطي الحاد.
    """
    if step < warmup_steps:
        return max_lr * (step / warmup_steps)

    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    progress = min(progress, 1.0)
    cosine_decay = 0.5 * (1 + np.cos(np.pi * progress))
    return min_lr + (max_lr - min_lr) * cosine_decay

def visualize_lr_schedule(total_steps=10000, warmup_steps=500, max_lr=3e-4):
    steps = np.arange(total_steps)
    lrs = [linear_warmup_cosine_decay(s, total_steps, warmup_steps, max_lr)
           for s in steps]

    plt.figure(figsize=(10, 4))
    plt.plot(steps, lrs)
    plt.axvline(x=warmup_steps, color='r', linestyle='--', label='نهاية Warmup')
    plt.xlabel("خطوة التدريب")
    plt.ylabel("معدل التعلّم")
    plt.title("جدولة Linear Warmup + Cosine Decay")
    plt.legend()
    plt.savefig("lr_schedule.png")
    print(f"LR عند step=0: {lrs[0]:.6f}")
    print(f"LR عند step=warmup_steps: {lrs[warmup_steps]:.6f}")
    print(f"LR عند step=total_steps/2: {lrs[total_steps//2]:.6f}")
    print(f"LR عند step=total_steps-1: {lrs[-1]:.6f}")

visualize_lr_schedule()
# LR عند step=0: 0.000000
# LR عند step=warmup_steps: 0.000300
# LR عند step=total_steps/2: 0.000150
# LR عند step=total_steps-1: 0.000000


### Regularization: منع الإفراط في التخصيص (Overfitting)


In [ ]:
import torch
import torch.nn as nn

class RegularizedMLP(nn.Module):
    """
    شبكة تطبّق أهم تقنيات Regularization الشائعة:
    Dropout + Weight Decay (عبر المُحسِّن) + Gradient Clipping (في التدريب)
    """
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.3):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.layer2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = torch.relu(self.layer1(x))
        x = self.dropout1(x)  # يُطفئ عشوائيًا نسبة من الخلايا أثناء التدريب فقط
        x = self.layer2(x)
        return x

def train_with_regularization(model, train_loader, epochs=10,
                                 max_grad_norm: float = 1.0):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()  # يُفعِّل Dropout
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()

            # Gradient Clipping: يمنع "انفجار التدرجات" (exploding gradients)
            # بتحديد أقصى قيمة لـ L2-norm الكلية لكل التدرجات مجتمعة
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)

            optimizer.step()

        model.eval()  # يُعطِّل Dropout أثناء التقييم (مهم جدًا!)


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: احسب يدويًا أول خطوتي تحديث لـ Adam على معامل واحد، وتحقق من تأثير تصحيح الانحياز (bias correction) في الخطوات الأولى.


In [ ]:
def manual_adam_steps():
    theta = 1.0       # قيمة ابتدائية للمعامل
    m, v = 0.0, 0.0   # العزمان الأول والثاني، يبدآن من الصفر
    beta1, beta2, lr, eps = 0.9, 0.999, 0.1, 1e-8
    gradients = [2.0, 1.8]  # تدرجات افتراضية لخطوتين متتاليتين

    for t, g in enumerate(gradients, start=1):
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        m_hat_no_correction = m   # بدون تصحيح انحياز (للتوضيح فقط)
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        update = lr * m_hat / (np.sqrt(v_hat) + eps)
        theta -= update

        print(f"الخطوة {t}: m={m:.4f}, m_hat (مُصحَّح)={m_hat:.4f} "
              f"(بدون تصحيح كان سيكون {m_hat_no_correction:.4f})")
        print(f"          v={v:.4f}, v_hat (مُصحَّح)={v_hat:.4f}")
        print(f"          تحديث θ: -{update:.4f} → θ الجديدة = {theta:.4f}\n")

manual_adam_steps()
# الخطوة 1: m=0.2000, m_hat (مُصحَّح)=2.0000 (بدون تصحيح كان سيكون 0.2000)
#           v=0.0040, v_hat (مُصحَّح)=4.0000
#           تحديث θ: -0.1000 → θ الجديدة = 0.9000
#
# الخطوة 2: m=0.3620, m_hat (مُصحَّح)=1.9053 (بدون تصحيح كان سيكون 0.3620)
#           v=0.0072, v_hat (مُصحَّح)=3.6076
#           تحديث θ: -0.1003 → θ الجديدة = 0.7997


> 🔴 لاحظ الفرق الهائل بين `m` الخام و`m_hat` المُصحَّح في الخطوة الأولى (0.2 مقابل 2.0!) — بدون تصحيح الانحياز، التحديثات الأولى ستكون ضعيفة جدًا وغير فعّالة لأن `m,v` ابتدأتا من الصفر.

---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم AdamW كخيار افتراضي | المعيار شبه العالمي لتدريب الشبكات العصبية والمحوّلات الحديثة |
| استخدم دائمًا Warmup لتدريب المحوّلات | بدونه، التدريب قد يتباعد (diverge) تمامًا في الخطوات الأولى، خصوصًا للنماذج الكبيرة |
| طبّق Gradient Clipping دائمًا | قيمة شائعة: `max_norm=1.0`، تحمي من الانفجار المفاجئ بسبب دفعة بيانات شاذة |
| لا تنسَ `model.eval()` قبل التقييم | Dropout وBatchNorm يتصرفان بشكل مختلف تمامًا بين التدريب والتقييم |


## 6. أخطاء شائعة ومصائد

- **استخدام نفس LR لكل أحجام النماذج**: النماذج الأكبر عادة تحتاج LR أصغر؛ قاعدة شائعة تجريبية: LR يتناسب عكسيًا تقريبًا مع الجذر التربيعي لعدد المعاملات.
- **نسيان `model.eval()`**: يُبقي Dropout فعّالًا أثناء التقييم، فتبدو النتائج أسوأ أو عشوائية بشكل غير متوقع.
- **الخلط بين L2 Regularization وWeight Decay في Adam**: كما شرحنا، إضافتهما بنفس الطريقة التقليدية (دمج L2 في التدرج) في Adam التقليدي يُعطي نتائج دون المستوى؛ AdamW يحل هذا بفصلهما.
- **Gradient Clipping بقيمة صغيرة جدًا**: يُبطئ التعلّم بشكل مصطنع؛ ابدأ بـ 1.0 واضبط تجريبيًا فقط إذا رأيت عدم استقرار.


## 7. جدول مقارنة

| المُحسِّن | الفكرة الجوهرية | الذاكرة الإضافية | الاستخدام الموصى به |
|---|---|---|---|
| SGD | تتبّع التدرج مباشرة | لا شيء إضافي | نماذج بسيطة جدًا، أو مع Momentum لرؤية حاسوبية كلاسيكية |
| SGD+Momentum | تراكم "زخم" الاتجاه | 1× حجم النموذج | رؤية حاسوبية تقليدية (ResNet إلخ) |
| Adam | تكيّف LR لكل معامل + Momentum | 2× حجم النموذج (m,v) | عام جدًا، نقطة انطلاق قياسية |
| **AdamW** | Adam + فصل صحيح لـ Weight Decay | 2× حجم النموذج | **معيار تدريب كل LLM حديث** |
| Lion | أبسط من Adam (يستخدم sign فقط) | 1× حجم النموذج (أقل ذاكرة) | بدائل حديثة توفر ذاكرة لنماذج ضخمة جدًا |


## 8. ملخص أفضل الممارسات

- استخدم AdamW مع `weight_decay≈0.01-0.1` كخيار افتراضي لتدريب المحوّلات.
- طبّق دائمًا Linear Warmup + Cosine Decay لجدولة LR في تدريب LLMs.
- استخدم Gradient Clipping (`max_norm=1.0`) كإجراء وقائي دائم.
- فعّل Dropout أثناء التدريب فقط، وتذكّر `.eval()`/`.train()` دائمًا.


## 9. أسئلة مقابلات

**س: ما الفرق الدقيق بين Adam وAdamW، ولماذا يُفضَّل الأخير لتدريب المحوّلات؟**

**ج**: في Adam التقليدي مع L2 regularization، يُضاف حد العقوبة (`λ·θ`) إلى **التدرج نفسه** قبل حساب `m` و`v`، مما يجعل تأثير الانحلال يُعاد تحجيمه (scaling) بواسطة `1/√v̂` التكيّفي لكل معامل — وهذا يُقلِّل فعاليته بشكل غير متوقع ومتفاوت للمعاملات المختلفة (المعاملات ذات التدرجات الكبيرة تاريخيًا تحصل على انحلال وزن أضعف فعليًا). AdamW يفصل هذين المسارين تمامًا: يُطبِّق التحديث التكيّفي القياسي (`m̂/√v̂`) على التدرج الأصلي فقط، ثم يُطبِّق انحلال الوزن (`λ·θ`) **مباشرة على الأوزان نفسها** بشكل منفصل تمامًا ومستقل عن أي تحجيم تكيّفي. هذا يجعل Weight Decay يعمل كما هو مُتوقَّع نظريًا (انكماش موحّد نحو الصفر)، وأظهرت التجارب تحسنًا ملموسًا في التعميم (generalization) لذلك أصبح المعيار القياسي.

**س: لماذا نحتاج Learning Rate Warmup عند تدريب المحوّلات الكبيرة تحديدًا؟**

**ج**: عدة أسباب متضافرة: (1) في بداية التدريب، تقديرات `m` و`v` في Adam مبنية على عدد قليل جدًا من العينات، فهي غير مستقرة إحصائيًا وقد تُنتج تحديثات كبيرة غير موثوقة. (2) الأوزان في بداية التدريب عشوائية تمامًا، والتدرجات الأولية قد تكون كبيرة وغير ممثِّلة للسلوك العام للدالة. (3) في معماريات المحوّلات تحديدًا، طبقات LayerNorm وAttention حساسة جدًا للتحديثات الكبيرة المبكرة، وقد تدخل الشبكة في حالة غير مستقرة (قيم logits متطرفة، انهيار تدريبي كامل) إذا بدأنا بـ LR الأقصى فورًا. Warmup يسمح للشبكة بـ"الاستقرار" تدريجيًا في منطقة معقولة من فضاء الأوزان قبل تطبيق خطوات تحديث كبيرة.


## 10. تمرين عملي

نفّذ `AdamWOptimizer` من الصفر بـ NumPy (بناءً على `AdamOptimizer` أعلاه)، ثم درّب `MLPFromScratch` من الفصل 3 على بيانات `make_moons` (من scikit-learn) باستخدام ثلاثة مُحسِّنات مختلفة (SGD، Adam، AdamW المخصَّص)، وارسم منحنيات الخسارة الثلاث على نفس الرسم البياني للمقارنة.


## 11. مراجع للتعمّق

- Kingma & Ba (2014) — *Adam: A Method for Stochastic Optimization*
- Loshchilov & Hutter (2017) — *Decoupled Weight Decay Regularization* (AdamW)
- Goyal et al. (2017) — *Accurate, Large Minibatch SGD* (أساس Linear Warmup)
- Srivastava et al. (2014) — *Dropout: A Simple Way to Prevent Overfitting*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — الأساسيات / الفصل 4: التحسين والتدريب: من SGD إلى AdamW.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
